In [1]:
!pip install -q transformers gradio sentencepiece

In [24]:
import torch
import gradio as gr
import re
from transformers import BartTokenizer, BartForConditionalGeneration

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [13]:
model_path = "/content/drive/MyDrive/nlp_summarization_project/bart_results/checkpoint-1250"

tokenizer = BartTokenizer.from_pretrained(model_path, local_files_only=True)
model = BartForConditionalGeneration.from_pretrained(model_path, local_files_only=True).to(device)
model.eval()

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [14]:
def clean_text(text: str) -> str:
    text = text.strip()
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\[\d+\]\s*", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text

In [25]:
def summarize_text(text):
    try:
        if not text or text.strip() == "":
            return "Please enter a text to summarize.", "", ""

        cleaned_text = clean_text(text)

        inputs = tokenizer(
            cleaned_text,
            return_tensors="pt",
            max_length=512,
            truncation=True
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=100,
                min_length=30,
                num_beams=4,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        stats = (
            f"Input word count: {len(cleaned_text.split())}\n"
            f"Summary word count: {len(summary.split())}"
        )

        return summary, cleaned_text, stats

    except Exception as e:
        err = f"ERROR: {str(e)}"
        return err, err, err

In [26]:
example_text = """With the rapid expansion of the Internet of Things (IoT) ecosystem, processing the data generated by billions of connected devices has started to place significant pressure on traditional cloud computing architectures. Although cloud-based systems provide high computational capacity, they also introduce critical limitations such as transmission latency, bandwidth constraints, and the need for continuous internet connectivity. Especially for applications that require real-time decision-making, processing data closer to its source has become increasingly necessary, making edge computing a practical and often essential approach.

In this context, Tiny Machine Learning (TinyML) has opened a new direction for IoT by enabling machine learning models to run directly on resource-constrained microcontrollers with ultra-low power consumption. By performing inference locally instead of sending raw data to external servers, TinyML solutions can reduce communication overhead and support stronger privacy through on-device processing. The literature emphasizes that this on-device intelligence supports low-latency responses and improves the feasibility of deploying AI in large-scale IoT environments."""

In [30]:
with gr.Blocks() as demo:
    gr.Markdown("# Transformer-Based Text Summarization")
    gr.Markdown("Paste a long text and generate a concise summary using the fine-tuned BART model.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                lines=14,
                label="Input Text",
                placeholder="Paste a article text or paragraph here..."
            )

            with gr.Row():
                summarize_btn = gr.Button("Summarize")
                example_btn = gr.Button("Load Example")
                clear_btn = gr.Button("Clear")

        with gr.Column():
            output_summary = gr.Textbox(
                lines=8,
                label="Generated Summary"
            )
            cleaned_preview = gr.Textbox(
                lines=8,
                label="Cleaned Input Used by Model"
            )
            stats_box = gr.Textbox(
                lines=3,
                label="Statistics"
            )

    summarize_btn.click(
        fn=summarize_text,
        inputs=[input_text],
        outputs=[output_summary, cleaned_preview, stats_box]
    )

    example_btn.click(
        fn=lambda: example_text,
        inputs=[],
        outputs=[input_text]
    )

    clear_btn.click(
        fn=lambda: ("", "", "", ""),
        inputs=[],
        outputs=[input_text, output_summary, cleaned_preview, stats_box]
    )

In [31]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2ade7fea8d200046c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
test_text = """With the rapid expansion of the Internet of Things (IoT) ecosystem, processing the data generated by billions of connected devices has started to place significant pressure on traditional cloud computing architectures. Although cloud-based systems provide high computational capacity, they also introduce critical limitations such as transmission latency, bandwidth constraints, and the need for continuous internet connectivity."""

result = summarize_text(test_text, "Medium")
print(result)

NameError: name 're' is not defined